# 03 - Cost-Sensitive Model with Threshold Optimization

Instead of optimizing for accuracy, we optimize for business cost:
- Missing a churner costs ₹5,000 (False Negative)
- Unnecessary discount costs ₹500 (False Positive)

We train a model and sweep the threshold from 0.05 to 0.95 to find the optimal cutoff.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from churnguard.config import TARGET_COLUMN
from churnguard.data.load import load_data
from churnguard.data.clean import clean_data
from churnguard.data.split import split_data
from churnguard.features.build_features import create_features
from churnguard.preprocessing.pipeline import build_pipeline
from churnguard.models.threshold_optimizer import optimize_threshold
from churnguard.models.cost_matrix import calculate_cost

# 1. Load, clean, split
df = clean_data(load_data())
X_train, X_test, y_train, y_test = split_data(df)

# 2. Pipeline
X_train_f = create_features(X_train)
X_test_f = create_features(X_test)
pipeline = build_pipeline(X_train_f)

X_train_tf = pipeline.fit_transform(X_train_f)
X_test_tf = pipeline.transform(X_test_f)

# 3. Train
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_tf, y_train)

# 4. Predict Probabilities
probs = clf.predict_proba(X_test_tf)[:, 1]

# 5. Optimize Threshold
best_threshold, best_cost, table = optimize_threshold(y_test, probs)
print(f"Optimal Threshold: {best_threshold}")
print(f"Cost at Optimal Threshold: ₹{best_cost}")

# 6. Compare to default 0.5 threshold
default_preds = (probs >= 0.5).astype(int)
default_cost = calculate_cost(y_test, default_preds)["Total Cost"]
print(f"Cost at default 0.5 threshold: ₹{default_cost}")
print(f"\nTotal Business Savings: ₹{default_cost - best_cost}")


2026-07-06 00:59:56,676 - churnguard.data.load - INFO - Loading raw dataset from /Users/himanshugahalyan/Desktop/CHURN_GUARD/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
2026-07-06 00:59:56,690 - churnguard.data.load - INFO - ==================================================
2026-07-06 00:59:56,691 - churnguard.data.load - INFO - Dataset Loaded Successfully
2026-07-06 00:59:56,691 - churnguard.data.load - INFO - ==================================================
2026-07-06 00:59:56,691 - churnguard.data.load - INFO - Shape : (7043, 21)
2026-07-06 00:59:56,691 - churnguard.data.load - INFO - Rows  : 7043
2026-07-06 00:59:56,692 - churnguard.data.load - INFO - Cols  : 21
2026-07-06 00:59:56,692 - churnguard.data.load - INFO - ==================================================
2026-07-06 00:59:56,692 - churnguard.data.clean - INFO - Starting data cleaning process...
2026-07-06 00:59:56,693 - churnguard.data.clean - INFO - Dropping customerID column.
2026-07-06 00:59:56,694 - churnguard.

Optimal Threshold: 0.07
Cost at Optimal Threshold: ₹390500
Cost at default 0.5 threshold: ₹1053000

Total Business Savings: ₹662500


By optimizing the threshold for real business costs rather than statistical accuracy, we save the business a significant amount of money.